# Подготовка аналитической витрины: нормализация и стандартизация данных

В первом комплекте был выполнен входной аудит данных. Теперь задача меняется: нужно не просто найти проблемы, а подготовить данные к расчёту управленческих показателей.

Рабочая ситуация: руководитель ждёт отчёт по продажам в разрезе регионов, каналов и категорий товаров. Перед отчётом нужно привести ключи, даты, категории и числовые показатели к единой логике.

Итогом работы станет аналитическая витрина `sales_datamart.csv` и сводная таблица `report_by_region_channel.csv`.

## 1. Подготовка окружения

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    candidates = [start, start.parent, start.parent.parent]
    for candidate in candidates:
        if (candidate / "data" / "raw").exists():
            return candidate
    raise FileNotFoundError("Не найдена папка data/raw. Откройте notebook из корня проекта или из папки notebooks.")

project_root = find_project_root()
data_dir = project_root / "data" / "raw"
output_dir = project_root / "outputs"
processed_dir = project_root / "data" / "processed"

output_dir.mkdir(exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)

print("Корень проекта:", project_root)
print("Папка данных:", data_dir)
print("Папка результатов:", output_dir)

## 2. Загрузка исходных данных

In [ ]:
sales = pd.read_csv(data_dir / "sales.csv")
products = pd.read_excel(data_dir / "products.xlsx")
clients = pd.read_csv(data_dir / "clients.csv")
regions = pd.read_json(data_dir / "regions.json")

print("sales:", sales.shape)
print("products:", products.shape)
print("clients:", clients.shape)
print("regions:", regions.shape)

## 3. Вспомогательные функции нормализации

В этом блоке мы готовим функции, которые будут повторяться в разных таблицах:

- убрать лишние пробелы;
- привести ключи к верхнему регистру;
- привести текстовые категории к нижнему регистру и удалить лишние пробелы внутри строки.

In [ ]:
def clean_text_series(series):
    return (
        series.astype("string")
        .str.strip()
        .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
    )

def clean_key_series(series):
    return clean_text_series(series).str.upper()

def normalize_label(series):
    return clean_text_series(series).str.lower().str.replace(r"\s+", " ", regex=True)

## 4. Нормализация таблицы продаж

In [ ]:
sales_work = sales.copy()

for col in ["order_id", "client_id", "product_id", "region_id"]:
    sales_work[col] = clean_key_series(sales_work[col])

sales_work["channel_raw"] = sales_work["channel"]
sales_work["channel"] = normalize_label(sales_work["channel"])
sales_work["channel"] = sales_work["channel"].replace({
    "marketplace": "marketplace",
    "online": "online",
    "retail": "retail",
})

sales_work["order_date_raw"] = sales_work["order_date"]
sales_work["order_date"] = pd.to_datetime(sales_work["order_date"], errors="coerce")
sales_work["quantity"] = pd.to_numeric(sales_work["quantity"], errors="coerce")
sales_work["unit_price"] = pd.to_numeric(sales_work["unit_price"], errors="coerce")
sales_work["discount"] = pd.to_numeric(sales_work["discount"], errors="coerce")

display(sales_work.head())
print("Каналы после нормализации:", sorted(sales_work["channel"].dropna().unique()))

## 5. Нормализация справочника товаров

In [ ]:
products_work = products.copy()
products_work["product_id"] = clean_key_series(products_work["product_id"])
products_work["product_name"] = clean_text_series(products_work["product_name"])
products_work["category"] = normalize_label(products_work["category"]).replace({
    "electronics": "Electronics",
    "accessories": "Accessories",
    "furniture": "Furniture",
    "office supplies": "Office Supplies",
    "appliances": "Appliances",
})
products_work["status"] = normalize_label(products_work["status"])
products_work["cost"] = pd.to_numeric(products_work["cost"], errors="coerce")

print("Дубли product_id до исправления:", products_work["product_id"].duplicated(keep=False).sum())
products_dedup = products_work.drop_duplicates(subset=["product_id"], keep="first").copy()
print("Строк после разрешения дублей:", len(products_dedup))
display(products_dedup.head())

## 6. Нормализация справочников клиентов и регионов

In [ ]:
clients_work = clients.copy()
clients_work["client_id"] = clean_key_series(clients_work["client_id"])
clients_work["segment"] = normalize_label(clients_work["segment"]).replace({
    "b2c": "B2C",
    "b2b": "B2B",
    "enterprise": "Enterprise",
    "smb": "SMB",
})
clients_work["loyalty_level"] = clean_text_series(clients_work["loyalty_level"]).str.title()
clients_work["registration_date"] = pd.to_datetime(clients_work["registration_date"], errors="coerce")

print("Дубли client_id до исправления:", clients_work["client_id"].duplicated(keep=False).sum())
clients_dedup = clients_work.drop_duplicates(subset=["client_id"], keep="first").copy()

regions_work = regions.copy()
regions_work["region_id"] = clean_key_series(regions_work["region_id"])
regions_work["region_name"] = clean_text_series(regions_work["region_name"])
regions_work["macro_region"] = clean_text_series(regions_work["macro_region"])
regions_dedup = regions_work.drop_duplicates(subset=["region_id"], keep="first").copy()

print("clients после разрешения дублей:", clients_dedup.shape)
print("regions после разрешения дублей:", regions_dedup.shape)

## 7. Флаги качества строк продаж

Здесь мы не удаляем строки сразу. Сначала создаём признаки, которые объясняют, почему строка может быть исключена из отчёта.

In [ ]:
valid_product_ids = set(products_dedup["product_id"].dropna())
valid_client_ids = set(clients_dedup["client_id"].dropna())
valid_region_ids = set(regions_dedup["region_id"].dropna())

sales_work["is_duplicate_order"] = sales_work["order_id"].duplicated(keep="first")
sales_work["invalid_date"] = sales_work["order_date"].isna()
sales_work["invalid_quantity"] = sales_work["quantity"].isna() | (sales_work["quantity"] <= 0)
sales_work["invalid_price"] = sales_work["unit_price"].isna() | (sales_work["unit_price"] <= 0)
sales_work["invalid_discount"] = sales_work["discount"].isna() | (sales_work["discount"] < 0) | (sales_work["discount"] > 1)
sales_work["missing_key"] = sales_work[["product_id", "client_id", "region_id"]].isna().any(axis=1)
sales_work["unknown_product"] = ~sales_work["product_id"].isin(valid_product_ids)
sales_work["unknown_client"] = ~sales_work["client_id"].isin(valid_client_ids)
sales_work["unknown_region"] = ~sales_work["region_id"].isin(valid_region_ids)

critical_flags = [
    "is_duplicate_order", "invalid_date", "invalid_quantity", "invalid_price",
    "invalid_discount", "missing_key", "unknown_product", "unknown_client", "unknown_region"
]

sales_work["exclude_from_report"] = sales_work[critical_flags].any(axis=1)
sales_work["quality_status"] = np.where(sales_work["exclude_from_report"], "excluded", "ready_for_report")

flag_summary = sales_work[critical_flags + ["exclude_from_report"]].sum().reset_index()
flag_summary.columns = ["check", "rows_count"]
display(flag_summary)

## 8. Подготовка строк, пригодных для отчёта

In [ ]:
sales_valid = sales_work[~sales_work["exclude_from_report"]].copy()

sales_valid["gross_revenue"] = sales_valid["quantity"] * sales_valid["unit_price"]
sales_valid["revenue"] = sales_valid["gross_revenue"] * (1 - sales_valid["discount"])

print("Строк в исходных продажах:", len(sales))
print("Строк, пригодных для отчёта:", len(sales_valid))
print("Исключено строк:", sales_work["exclude_from_report"].sum())
display(sales_valid.head())

## 9. Сборка аналитической витрины

Теперь объединяем продажи со справочниками. Для защиты от скрытого размножения строк используем проверку типа соединения `many_to_one`: много строк продаж могут ссылаться на один товар, одного клиента или один регион.

In [ ]:
sales_datamart = (
    sales_valid
    .merge(products_dedup[["product_id", "product_name", "category", "cost"]], on="product_id", how="left", validate="many_to_one")
    .merge(clients_dedup[["client_id", "segment", "loyalty_level", "registration_date"]], on="client_id", how="left", validate="many_to_one")
    .merge(regions_dedup[["region_id", "region_name", "macro_region"]], on="region_id", how="left", validate="many_to_one")
)

sales_datamart["margin"] = sales_datamart["revenue"] - sales_datamart["quantity"] * sales_datamart["cost"]
sales_datamart["order_month"] = sales_datamart["order_date"].dt.to_period("M").astype(str)

print("Строк в витрине:", len(sales_datamart))
display(sales_datamart.head())

## 10. Стандартизация и min-max нормализация выручки

Z-score показывает, насколько заказ отличается от среднего заказа в стандартных отклонениях.

Min-max нормализация переводит выручку в диапазон от 0 до 1 и удобна для сопоставления значений на общей шкале.

In [ ]:
revenue_mean = sales_datamart["revenue"].mean()
revenue_std = sales_datamart["revenue"].std(ddof=0)
revenue_min = sales_datamart["revenue"].min()
revenue_max = sales_datamart["revenue"].max()

sales_datamart["revenue_zscore"] = (sales_datamart["revenue"] - revenue_mean) / revenue_std
sales_datamart["revenue_minmax"] = (sales_datamart["revenue"] - revenue_min) / (revenue_max - revenue_min)

display(sales_datamart[["order_id", "revenue", "revenue_zscore", "revenue_minmax"]].head())
print("Среднее z-score:", round(sales_datamart["revenue_zscore"].mean(), 6))
print("min revenue_minmax:", sales_datamart["revenue_minmax"].min())
print("max revenue_minmax:", sales_datamart["revenue_minmax"].max())

## 11. Итоговая сводка для отчёта

In [ ]:
report_by_region_channel = (
    sales_datamart
    .groupby(["macro_region", "region_name", "channel"], as_index=False)
    .agg(
        orders_count=("order_id", "nunique"),
        total_quantity=("quantity", "sum"),
        total_revenue=("revenue", "sum"),
        avg_order_revenue=("revenue", "mean"),
        total_margin=("margin", "sum")
    )
    .sort_values(["total_revenue"], ascending=False)
)

display(report_by_region_channel.head(10))

## 12. Лог трансформаций

In [ ]:
transformation_log = pd.DataFrame([
    {"step": "raw_sales_rows", "value": len(sales), "comment": "Строк в исходной таблице продаж"},
    {"step": "duplicate_orders_excluded", "value": int(sales_work["is_duplicate_order"].sum()), "comment": "Повторные order_id исключены из витрины"},
    {"step": "invalid_dates_excluded", "value": int(sales_work["invalid_date"].sum()), "comment": "Строки с нераспознанной датой исключены"},
    {"step": "invalid_quantity_excluded", "value": int(sales_work["invalid_quantity"].sum()), "comment": "Строки с некорректным количеством исключены"},
    {"step": "invalid_price_excluded", "value": int(sales_work["invalid_price"].sum()), "comment": "Строки с некорректной ценой исключены"},
    {"step": "invalid_discount_excluded", "value": int(sales_work["invalid_discount"].sum()), "comment": "Строки с некорректной скидкой исключены"},
    {"step": "sales_ready_for_report", "value": len(sales_datamart), "comment": "Строк готовой витрины"},
])

display(transformation_log)

## 13. Сохранение результатов

In [ ]:
sales_work.to_csv(output_dir / "sales_prepared_with_quality_flags.csv", index=False)
sales_datamart.to_csv(output_dir / "sales_datamart.csv", index=False)
report_by_region_channel.to_csv(output_dir / "report_by_region_channel.csv", index=False)
transformation_log.to_csv(output_dir / "transformation_log.csv", index=False)

print("Файлы сохранены в:", output_dir)
for file_path in sorted(output_dir.glob("*.csv")):
    print("-", file_path.name)

## 14. Решение аналитика

Заполните итоговое решение в файле `normalization_decision.md` или отдельной markdown-ячейке.

Шаблон:

```text
1. Что было сделано с данными:
2. Сколько строк осталось в готовой витрине:
3. Какие строки исключены из отчёта и почему:
4. Какие ограничения нужно указать руководителю:
5. Какие файлы можно использовать для следующего отчётного блока:
```